## **Install Library**

In [ ]:
import shutil, os

CHROMA_DIR = "/content/chatbot-db/chroma_db"
GDRIVE_DIR = "/gdrive/MyDrive/chatbot-db/chroma_db"

if not os.path.exists(CHROMA_DIR) and os.path.exists(GDRIVE_DIR):
    os.makedirs(os.path.dirname(CHROMA_DIR), exist_ok=True)
    shutil.copytree(GDRIVE_DIR, CHROMA_DIR)
    print("Copied ChromaDB from GDrive")
else:
    print("ChromaDB already exists at", CHROMA_DIR)

In [ ]:
!pip install -q transformers accelerate bitsandbytes gradio
!pip install -q langchain-text-splitters
!pip install -q langchain langchain-community chromadb pypdf sentence-transformers
!pip install -q huggingface_hub
!pip install -q fastapi uvicorn nest-asyncio
!pip install -q python-multipart

## **Cek GPU**

In [ ]:
import torch

print("CUDA tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU aktif bang:", torch.cuda.get_device_name(0))
else:
    print("GPU tidak aktif. Aktifkan GPU bang")

## **Import Library dan login Hugging Face**

In [ ]:
import torch
import gradio as gr

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HF_TOKEN = userdata.get("HF")
login(token=HF_TOKEN)

print("Login Hugging Face berhasil.")

## **Menyimpan PDF permanen ke Gdrive**

In [ ]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

## **Cek PDF di Gdrive**

In [ ]:
import os

PDF_DIR = "/gdrive/MyDrive/chatbot-pdfs"

pdf_files = [
    os.path.join(PDF_DIR, f)
    for f in os.listdir(PDF_DIR)
    if f.lower().endswith(".pdf")
]

print(f"Jumlah PDF ditemukan: {len(pdf_files)}")

for pdf in pdf_files:
    print("-", os.path.basename(pdf))

## **Load isi PDF dan Filter halaman yang punya teks**

In [ ]:
all_documents = []

for pdf_path in pdf_files:
    print(f"Loading {os.path.basename(pdf_path)}")

    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    all_documents.extend(docs)

documents = all_documents

documents_clean = []

for doc in documents:
    if doc.page_content.strip():
        documents_clean.append(doc)

print("Jumlah halaman asli:", len(documents))
print("Jumlah halaman berisi teks:", len(documents_clean))

print("Contoh isi dokumen bersih:")
print(documents_clean[0].page_content[:500])

## **Chungking Dokumen**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents_clean)

print("Jumlah chunk:", len(chunks))
print("Contoh chunk:")
print(chunks[0].page_content[:500])

## **Embending**

In [ ]:
EMBEDDING_MODEL = "BAAI/bge-m3"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model siap:", EMBEDDING_MODEL)

## **Menyimpan Hasil ke ChromaDB**

In [ ]:
import os
import glob
import json
import threading

CHROMA_DIR = "/content/chatbot-db/chroma_db"
os.makedirs(CHROMA_DIR, exist_ok=True)

PDF_DIR = "/gdrive/MyDrive/chatbot-pdfs/"
os.makedirs(PDF_DIR, exist_ok=True)

vector_db_lock = threading.Lock()
inference_lock = threading.Lock()

MANIFEST_PATH = os.path.join(PDF_DIR, "_processed_manifest.json")

def load_manifest():
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            return set(json.load(f))
    return set()

def save_manifest(processed_set):
    with open(MANIFEST_PATH, "w") as f:
        json.dump(sorted(processed_set), f, indent=2)

processed_pdfs = load_manifest()

db_exists = os.path.exists(os.path.join(CHROMA_DIR, "chroma.sqlite3"))

vector_db = Chroma(
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR
)

if db_exists:
    print("ChromaDB loaded dari Google Drive (persisten).")
else:
    print("ChromaDB baru dibuat di Google Drive.")

all_pdfs_in_dir = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
new_pdfs = [p for p in all_pdfs_in_dir if os.path.basename(p) not in processed_pdfs]

if not new_pdfs:
    print("Tidak ada PDF baru untuk diproses, semua sudah ada di ChromaDB.")
else:
    for pdf_path in new_pdfs:
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        docs_clean = [d for d in docs if d.page_content.strip()]
        pdf_chunks = text_splitter.split_documents(docs_clean)

        with vector_db_lock:
            vector_db.add_documents(pdf_chunks)
            processed_pdfs.add(os.path.basename(pdf_path))

        print(f"  -> {os.path.basename(pdf_path)}: {len(pdf_chunks)} chunk ditambahkan")

    save_manifest(processed_pdfs)

retriever = vector_db.as_retriever(search_kwargs={"k": 15})
print("Retriever aktif. Total PDF tercatat di manifest:", len(processed_pdfs))

## **Mengambil data dari PDF/Dokumen**

In [ ]:
from pathlib import Path

def get_pdf_context(query, source_filter=None):
    try:
        docs = retriever.invoke(query)

        if source_filter:
            filtered = [d for d in docs if source_filter in Path(d.metadata.get('source', '')).name]
            if filtered:
                docs = filtered

        if not docs:
            return "Tidak ada konteks PDF yang relevan ditemukan."

        context = "\n\n".join([
            f"[Dokumen: {Path(d.metadata.get('source', 'unknown')).name}] Halaman {d.metadata.get('page', '?')}:\n{d.page_content}"
            for d in docs
        ])

        return context

    except Exception as e:
        return f"Gagal mengambil konteks dari PDF: {str(e)}"


def detect_topic(query):
    text = query.lower()

    topic_map = {
        "dasar-dasar pemrograman": ["java", "tipe data primitif", "identifier", "class", "komentar", "statement", "blok", "modulo", "operator aritmetika", "operator logika", "operator kondisi"],
        "pythonlearn": ["instalasi python", "tipe data dasar", "penugasan", "perbandingan", "if elif else", "for", "while", "range", "list", "tuple", "dictionary", "function", "return", "exception", "try except", "file teks", "class", "object", "__init__", "interpreter", "compiler", "variable name", "string slicing", "regular expression", "findall"],
        "progit": ["git", "snapshot", "vcs", "subversion", "svn", "commit", "branch", "merge conflict", "fetch", "pull", "rebase", "tag", "annotated tag", "github flow", "pull request", "stash", "bisect"],
        "mysqlnotes": ["mysql", "create database", "char", "varchar", "text", "distinct", "like", "wildcard", "join", "left join", "right join", "on duplicate key", "user-defined variable", "index", "foreign key", "error 1215", "date range"],
        "pemrograman web dasar": ["html", "css", "javascript", "php", "client side", "server side", "tabel html", "include", "require", "layout website"],
        "buku-panduan-genai": ["genai", "generative ai", "artificial intelligence", "tuce", "t.u.c.e.", "refleksi kritis", "bobot toleransi", "plagiarisme", "sanksi"],
        "pemrograman komputer full": ["return", "tanpa return", "__init__", "modul", "package", "import", "mode r", "mode w", "mode a", "variabel global", "variabel lokal"]
    }

    for topic, keywords in topic_map.items():
        for kw in keywords:
            if kw in text:
                return topic

    return None

In [ ]:
print("=" * 60)
print("ISI PER PDF (dari ChromaDB)")
print("=" * 60)

all_docs = vector_db.get()
sources = {}
for doc_id, meta in zip(all_docs["ids"], all_docs["metadatas"]):
    src = os.path.basename(meta.get("source", "unknown"))
    if src not in sources:
        sources[src] = {"chunks": 0, "pages": set()}
    sources[src]["chunks"] += 1
    sources[src]["pages"].add(meta.get("page", "?"))

for pdf_name in sorted(processed_pdfs):
    info = sources.get(pdf_name, {"chunks": 0, "pages": set()})
    print(f" -  {pdf_name}: {info['chunks']} chunk dari {len(info['pages'])} halaman")

## **Sistem Prompt**

In [ ]:
SYSTEM_PROMPT = """
Kamu adalah chatbot Generative AI sederhana berbasis NLP.

Tugas utama:
1. Menjawab pertanyaan umum dengan bahasa Indonesia yang jelas dan mudah dipahami.
2. Membantu coding ringan, terutama Python, HTML, CSS, JavaScript, dan error dasar.
3. Menjelaskan konsep programming untuk pemula.
4. Memberikan contoh kode sederhana jika dibutuhkan.
5. Jika tidak yakin, katakan bahwa kamu tidak yakin.
6. Jangan mengarang fakta yang terlalu spesifik.

Aturan jawaban:
- Jawab dalam bahasa Indonesia.
- Jawaban harus singkat, jelas, dan langsung ke inti.
- Untuk pertanyaan coding, jelaskan penyebab masalah dan berikan solusi.
- Jika memberi kode, gunakan kode yang sederhana dan mudah dipahami.
- Jangan membuat jawaban terlalu panjang kalau tidak diminta.
"""

CUSTOM_KNOWLEDGE = """
Informasi tentang chatbot ini:
- Chatbot ini dibuat menggunakan Google Colab.
- Model AI dijalankan menggunakan Hugging Face Transformers.
- Backend dibuat menggunakan FastAPI.
- Frontend dibuat menggunakan React dan TailwindCSS.
- Backend Colab dihubungkan ke frontend lokal menggunakan Cloudflare Tunnel.
- Chatbot ini ditujukan untuk menjawab pertanyaan umum dan membantu coding ringan.
"""

RAG_SYSTEM_PROMPT = """
Kamu adalah chatbot RAG yang menjawab BERDASARKAN konteks PDF.

Aturan WAJIB:
1. Jawab HANYA berdasarkan konteks PDF yang diberikan.
2. Perhatikan label [Dokumen: ...] di setiap potongan konteks. Jawab SESUAI dokumen yang relevan dengan pertanyaan.
3. JANGAN gunakan informasi dari dokumen yang tidak relevan dengan pertanyaan.
4. JANGAN campur informasi dari dokumen berbeda dalam satu jawaban.
5. Jika ada kontradiksi antar dokumen, utamakan yang sesuai topik pertanyaan.
6. Jika ditanya tentang isi dokumen tertentu, gunakan konteks PDF yang sudah diberikan.
7. Jika konteks cukup, jawab dengan jelas dan langsung.
8. Jika konteks tidak cukup, akui: "Dari dokumen yang tersedia, saya tidak menemukan informasi tentang..."
9. Jawab dalam bahasa Indonesia dengan jelas.
"""

mengtes spek colep buat model

In [ ]:
!nvidia-smi

## **Load Model General Knowlage**

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config
)

model.eval()

print("Model berhasil diload.")

## **Fungsi Untuk Generate Jawaban**

In [ ]:
def build_rag_messages(user_message, history):
    intent = detect_intent(user_message)
    topic = detect_topic(user_message)

    pdf_context = get_pdf_context(user_message, source_filter=topic)
    daftar_pdf = "\n".join(f"- {p}" for p in sorted(processed_pdfs))
    pdf_context += f"\n\nDaftar dokumen yang tersedia:\n{daftar_pdf}"

    intent_instruction = f"""
Intent pengguna terdeteksi: {intent}
Topik terdeteksi: {topic or 'umum'}

Gunakan intent ini untuk membantu menjawab:
- coding_error: jelaskan penyebab error dan solusi
- coding_help: bantu dengan contoh kode sederhana
- general_question: jawab konsep umum dengan jelas
- general_chat: jawab secara natural
"""

    messages = [
        {
            "role": "system",
            "content": (
                "Konteks dari PDF:\n" + pdf_context
                + "\n\n"
                + RAG_SYSTEM_PROMPT
                + "\n\n"
                + CUSTOM_KNOWLEDGE
                + "\n\n"
                + intent_instruction
            )
        }
    ]

    for chat in history:
        if chat["role"] in ["user", "assistant"]:
            messages.append({
                "role": chat["role"],
                "content": chat["content"]
            })

    messages.append({
        "role": "user",
        "content": user_message
    })

    return messages

In [ ]:
def detect_intent(user_message):
    text = user_message.lower()
    if any(word in text for word in ["error", "bug", "syntax", "nameerror", "modulenotfound", "traceback"]):
        return "coding_error"
    elif any(word in text for word in ["python", "html", "css", "javascript", "kode", "function", "array", "variabel"]):
        return "coding_help"
    elif any(word in text for word in ["apa itu", "jelaskan", "pengertian", "definisi", "konsep"]):
        return "general_question"
    else:
        return "general_chat"


def generate_response_rag(user_message, history):
    messages = build_rag_messages(user_message, history)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with inference_lock:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=350,
                do_sample=True,
                temperature=0.3,
                top_p=0.9,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id
            )

    input_length = inputs["input_ids"].shape[-1]
    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    answer = answer.replace("<|im_end|>", "").strip()
    return answer

test

In [ ]:
question = "Jelaskan isi utama dari dokumen GenAI untuk mahasiswa"

answer = generate_response_rag(
    question,
    history=[]
)

print(answer)

## **Buat Backend FastAPI**

In [ ]:
import queue
import asyncio
from threading import Thread
from transformers import TextStreamer

class AsyncStreamer(TextStreamer):
    def __init__(self, tokenizer, skip_prompt=True, skip_special_tokens=True):
        super().__init__(tokenizer, skip_prompt=skip_prompt, skip_special_tokens=skip_special_tokens)
        self.token_queue = queue.Queue()
        self.generation_finished = False

    def on_finalized_text(self, text: str, stream_end: bool = False):
        if stream_end:
            self.token_queue.put(None)
            self.generation_finished = True
        elif text:
            self.token_queue.put(text)

    async def get_token(self):
        while True:
            try:
                token = self.token_queue.get_nowait()
                return token
            except queue.Empty:
                if self.generation_finished:
                    return None
                await asyncio.sleep(0.01)

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Literal
import nest_asyncio
import uvicorn
import threading
import aiofiles
import os
from fastapi import UploadFile, File
from starlette.responses import StreamingResponse
import asyncio

api_app = FastAPI()

api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatMessage(BaseModel):
    role: Literal["user", "assistant"]
    content: str

class ChatRequest(BaseModel):
    message: str
    history: List[ChatMessage] = []

@api_app.get("/")
def home():
    return {
        "message": "FastAPI chatbot backend baru jalan."
    }

@api_app.get("/api/ping")
def ping():
    return {
        "status": "ok",
        "message": "Backend baru bisa diakses."
    }

@api_app.post("/api/chat")
def chat(request: ChatRequest):
    user_message = request.message.strip()

    if not user_message:
        return {
            "reply": "Pesannya kosong. Ketik sesuatu dulu."
        }

    try:
        history = [
            msg.model_dump() if hasattr(msg, "model_dump") else msg.dict()
            for msg in request.history
        ]

        answer = generate_response_rag(
            user_message,
            history=history
        )

        return {
            "reply": answer
        }

    except Exception as e:
        return {
            "reply": f"Terjadi error di backend: {str(e)}"
        }

MAX_PDF_SIZE_MB = 20
UPLOAD_DIR = PDF_DIR

@api_app.post("/api/chat/stream")
async def chat_stream(request: ChatRequest):
    user_message = request.message.strip()

    if not user_message:
        async def empty_error():
            yield f"data: {json.dumps({'error': 'Pesannya kosong. Ketik sesuatu dulu.'})}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(empty_error(), media_type="text/event-stream")

    try:
        history_raw = [
            msg.model_dump() if hasattr(msg, "model_dump") else msg.dict()
            for msg in request.history
        ]

        messages = build_rag_messages(user_message, history_raw)

        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        streamer = AsyncStreamer(
            tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        async def generate():
            def run_model():
                with inference_lock:
                    with torch.no_grad():
                        model.generate(
                            **inputs,
                            max_new_tokens=350,
                            do_sample=True,
                            temperature=0.3,
                            top_p=0.9,
                            repetition_penalty=1.05,
                            pad_token_id=tokenizer.eos_token_id,
                            streamer=streamer,
                        )

            loop = asyncio.get_event_loop()
            future = loop.run_in_executor(None, run_model)

            while True:
                token = await streamer.get_token()
                if token is None:
                    break
                yield f"data: {json.dumps({'token': token})}\n\n"

            await future
            yield "data: [DONE]\n\n"

        return StreamingResponse(generate(), media_type="text/event-stream")

    except Exception as e:
        async def error_stream():
            yield f"data: {json.dumps({'error': f'Terjadi error di backend: {str(e)}'})}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(error_stream(), media_type="text/event-stream")

## **Menjakankan FastAPI di Google Colab**

In [ ]:
nest_asyncio.apply()

def run_api_8010():
    uvicorn.run(
        api_app,
        host="0.0.0.0",
        port=8010
    )

thread = threading.Thread(target=run_api_8010)
thread.start()

print("FastAPI baru jalan di port 8010")

## **Mengambil URL API Colab**

In [ ]:
from google.colab.output import eval_js

api_url = eval_js("google.colab.kernel.proxyPort(8010)")
print("API URL:", api_url)

## **Test BE Colab**

In [ ]:
import requests

res = requests.get("http://127.0.0.1:8010/api/ping")
print(res.status_code)
print(res.json())

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import subprocess
import time
import re

process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8010"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None

for _ in range(60):
    line = process.stdout.readline()
    print(line, end="")

    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

    time.sleep(1)

print("\nPUBLIC URL:", public_url)

## **Test chat apakah sesuai dgn rag**

In [ ]:
res = requests.post(
    "http://127.0.0.1:8010/api/chat",
    json={
        "message": "Jelaskan apa isi dari dokumen progit?",
        "history": []
    }
)

print(res.status_code)
print(res.json())

## **Test Via Notebook**

In [ ]:
context = get_pdf_context("Apa isi dokumen Progit bab 1")

print(context[:1500])

## **Eval ROUGE**

In [ ]:
!pip install -q rouge-score